In [ ]:
from config import init_env
from config import variablesc
import importlib
variables = importlib.reload(variables)
init_env.set_environment_variables()


In [18]:
import bs4
from langchain_community.document_loaders import WebBaseLoader

# Only keep post title, headers, and content from the full HTML.
bs4_strainer = bs4.SoupStrainer(class_=("post-title", "post-header", "post-content"))
loader = WebBaseLoader(
    web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs={"parse_only": bs4_strainer},
)
docs = loader.load()

assert len(docs) == 1
print(f"Total characters: {len(docs[0].page_content)}")

Total characters: 43047


In [28]:
import bs4
from langchain_community.document_loaders import WebBaseLoader

# Only keep post title, headers, and content from the full HTML.
bs4_strainer = bs4.SoupStrainer(class_=("MessageSubject","lia-message-body-content"))
loader = WebBaseLoader(
    web_paths=("https://community.sap.com/t5/technology-blog-posts-by-sap/abap-development-tools-for-vs-code-everything-you-need-to-know/ba-p/14258129",),
    bs_kwargs={"parse_only": bs4_strainer},
)
docs = loader.load()

assert len(docs) == 1
print(f"Total characters: {len(docs[0].page_content)}")

Total characters: 3981


In [8]:
import bs4
from langchain_community.document_loaders import WebBaseLoader

# Only keep post title, headers, and content from the full HTML.
bs4_strainer = bs4.SoupStrainer(class_=("  page-content border-top entry-content","content-wrapper"))
loader = WebBaseLoader(
    web_paths=("https://www.nobelprize.org/prizes/physics/2025/press-release/",),
    bs_kwargs={"parse_only": bs4_strainer},
)
docs = loader.load()

assert len(docs) == 1
print(f"Total characters: {len(docs[0].page_content)}")

Total characters: 5586


In [9]:
print(docs[0].page_content[:5000])




				Press release			


Navigate to:
 Summary- John Clarke- Michel H. Devoret- John M. Martinis Prize announcement Press release Advanced information Popular information
EnglishEnglish (pdf)SwedishSwedish (pdf)

7 October 2025
The Royal Swedish Academy of Sciences has decided to award the Nobel Prize in Physics 2025 to
John ClarkeUniversity of California, Berkeley, USA
Michel H. DevoretYale University, New Haven, CT andUniversity of California, Santa Barbara, USA
John M. MartinisUniversity of California, Santa Barbara, USA and Qolab, Los Angeles, CA, USA
“for the discovery of macroscopic quantum mechanical tunnelling and energy quantisation in an electric circuit”
Their experiments on a chip revealed quantum physics in action
A major question in physics is the maximum size of a system that can demonstrate quantum mechanical effects. This year’s Nobel Prize laureates conducted experiments with an electrical circuit in which they demonstrated both quantum mechanical tunnelling and quan

In [10]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,  # chunk size (characters)
    chunk_overlap=300,  # chunk overlap (characters)
    add_start_index=True,  # track index in original document
)
all_splits = text_splitter.split_documents(docs)

print(f"Split blog post into {len(all_splits)} sub-documents.")

Split blog post into 9 sub-documents.


In [3]:
# Step 3: Set embedding model  
 
from gen_ai_hub.proxy.langchain.openai import ChatOpenAI, OpenAIEmbeddings
embeddings = OpenAIEmbeddings(deployment_id=variables.EMBEDDING_DEPLOYMENT_ID)  # Deployment ID of text-embedding-3-large

In [4]:
from langchain_chroma import Chroma

vector_store = Chroma(
    collection_name="NobelPrize",
    embedding_function=embeddings,
    persist_directory="./chroma_db",  # Where to save data locally, remove if not necessary
)

In [21]:
document_ids = vector_store.add_documents(documents=all_splits)
for i in document_ids:
    print(i)

658ecaba-b0c1-4636-bbd2-006ddaa538b9
5690bc14-199c-402e-bc25-42c3eb5830dd
5e9df78a-3d3e-4d5a-80c5-cdae05847efd
37945564-9117-448e-8dac-194fd31c398e
79b80593-e6aa-4759-9636-d7f0936c6e01
6eec8500-d039-476b-885c-06a8c6553806
f8b02b25-8709-45b6-a599-4959959e825d
2c2d9114-28fc-42d8-88ff-2e19c2c76607
934ee7ec-9bac-4021-b4b8-e6005f3b31ba


In [23]:
# Search by vector
import json

from gen_ai_hub.proxy.langchain.openai import ChatOpenAI, OpenAIEmbeddings
embedding_model = OpenAIEmbeddings(deployment_id=variables.EMBEDDING_DEPLOYMENT_ID)  # Embedding deployment ID 
 
search_results=vector_store.similarity_search_by_vector(
    embedding=embedding_model.embed_query(
        text="When was John M. Martinis born?"),  
    k=1)

# print results
i=1
for r in search_results:
    # print(f"Content:{r.page_content}") 
    print("————————————————————————————————————","search result",i,"—————————————————————————————————————")
    print(f"Content: {r.page_content}")
    print("metadata:")
    print(json.dumps(r.metadata, indent=3))
    i=i+1
 

———————————————————————————————————— search result 1 —————————————————————————————————————
Content: John Clarke, born 1942 in Cambridge, UK. PhD 1968 from University of Cambridge, UK. Professor at University of California, Berkeley, USA.
Michel H. Devoret, born 1953 in Paris, France. PhD 1982 from Paris-Sud University, France. Professor at Yale University, New Haven, CT and University of California, Santa Barbara, USA.
John M. Martinis, born 1958. PhD 1987 from University of California, Berkeley, USA. Professor at University of California, Santa Barbara, USA and Chief Technology Officer at Qolab, Los Angeles, CA, USA.

Prize amount: 11 million Swedish kronor, to be shared equally between the laureates.Further information: www.kva.se and www.nobelprize.orgPress contact: Eva Nevelius, Press Secretary, +46 70 878 67 63, [email protected]Experts: Göran Johansson, +46 31 772 32 37, [email protected] and Eva Lindroth, +46 8 553 786 16, [email protected], members of the Nobel Committee for Ph

In [8]:
# Define prompt (to avoid hallucination)
from langchain_core.prompts import PromptTemplate
template = """
You are an AI assistant. Answer the question only based on the provided context.
If the answer is not contained in the context, say "The document does not contain this information."

Context:
{context}

Question:
{question}
"""

prompt = PromptTemplate(
    input_variables=["context", "question"],
    template=template,
)

# Initialize LLM
chat_llm = ChatOpenAI(deployment_id=variables.LLM_DEPLOYMENT_ID)  # LLM deployment ID. Here gpt-4o has been maintained



# Define format function
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser


# use "|" operator to compose runnables together in a pipeline. 
def format_docs(docs):
    return "\n".join(doc.page_content for doc in docs)

rag_chain=(
    {
        "context":vector_store.as_retriever(search_kwargs={"k": 5}) | format_docs,  #1 Build a dictionary with context and question.
        "question":RunnablePassthrough()
    }
    | prompt                                                                        #2 Pass that dictionary to prompt (which formats it into text).
    | chat_llm                                                                      #3 Send the formatted prompt to chat_llm (the language model).
    | StrOutputParser ()                                                            #4 parse the output into a string with StrOutputParser().
)


# Define qa function
def qa(question):
    response = rag_chain.invoke(question)
    print ("Question: ",question)
    print ("Answer: ",response)
    print("———————————————————————————————————————————————————————————————————————")

In [9]:
qa("Who won the Nobel Prize of Physics in 2025?")

Question:  Who won the Nobel Prize of Physics in 2025?
Answer:  John Clarke, Michel H. Devoret, and John M. Martinis won the Nobel Prize in Physics in 2025.
———————————————————————————————————————————————————————————————————————


In [10]:
qa("Who won the Nobel Prize of Physics in 2023?")

Question:  Who won the Nobel Prize of Physics in 2023?
Answer:  The document does not contain this information.
———————————————————————————————————————————————————————————————————————
